# LeetCode #120 — Triangle: DP từ đáy lên, bộ nhớ O(n)

**Mức độ:** Medium · **Chủ đề:** Array, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def minimumTotal(self, triangle: List[List[int]]) -> int:
```

Mục tiêu của notebook:

1. Mô hình hóa: đi từ đỉnh xuống, ở vị trí `j` chỉ được xuống `j` hoặc `j + 1`.
2. DP **từ trên xuống** — và các trường hợp biên khó chịu.
3. DP **từ đáy lên** — không có biên, đáp án nằm ở một ô duy nhất.
4. Follow-up: bộ nhớ `O(n)` (n = số hàng), thậm chí sửa tại chỗ.

# 1. Bài toán là gì?

Cho tam giác số `triangle` (hàng `i` có `i + 1` phần tử). Tìm tổng **nhỏ nhất** của một đường đi từ **đỉnh** xuống **đáy**. Ở mỗi bước, từ vị trí `j` của hàng `i`, được đi xuống vị trí `j` hoặc `j + 1` của hàng `i + 1`.

**Constraints (theo đề):**

- `1 <= triangle.length <= 200`
- `triangle[0].length == 1`, `triangle[i].length == triangle[i − 1].length + 1`
- `-10^4 <= triangle[i][j] <= 10^4`

**Follow up (theo đề):** chỉ dùng `O(n)` bộ nhớ phụ, với `n` là số hàng.

---

## Ví dụ

```text
      2
     3 4
    6 5 7
   4 1 8 3        ->  11      (2 + 3 + 5 + 1)

triangle = [[-10]]  ->  -10
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 Áp dụng vào bài này

**Quy hoạch động** từ **đáy lên**: `dp[j]` = tổng nhỏ nhất từ ô `(i, j)` xuống đáy = `triangle[i][j] + min(dp[j], dp[j+1])` của hàng dưới; cập nhật ngay trên một mảng một chiều.

# 3. Tham lam có đúng không?

Mỗi bước chọn ô nhỏ hơn trong hai ô bên dưới:

In [3]:
from typing import List
from functools import lru_cache
import random

def greedy(triangle: List[List[int]]) -> int:
    j, total = 0, triangle[0][0]
    for i in range(1, len(triangle)):
        if triangle[i][j + 1] < triangle[i][j]:
            j += 1
        total += triangle[i][j]
    return total

EX = [[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]
print("Tham lam:", greedy(EX), "(đúng: 11)")
T = [[1], [2, 3], [9, 9, 1], [9, 9, 9, 1]]
print("Tham lam:", greedy(T), "(đúng: 1 + 3 + 1 + 1 = 6)")

Tham lam: 11 (đúng: 11)
Tham lam: 21 (đúng: 1 + 3 + 1 + 1 = 6)


Ví dụ thứ hai: tham lam chọn `2` (nhỏ hơn `3`), rồi kẹt giữa các số `9`. Phải nhìn xa → DP.

# 4. Baseline: đệ quy

`f(i, j)` = tổng nhỏ nhất từ ô `(i, j)` xuống đáy = `triangle[i][j] + min(f(i+1, j), f(i+1, j+1))`.

Không memo: `2^(n−1)` đường đi. Có memo: chỉ `n(n+1)/2` ô.

In [4]:
def min_total_memo(triangle: List[List[int]]) -> int:
    n = len(triangle)
    @lru_cache(maxsize=None)
    def f(i, j):
        if i == n - 1:
            return triangle[i][j]
        return triangle[i][j] + min(f(i + 1, j), f(i + 1, j + 1))
    return f(0, 0)

print(min_total_memo(EX), min_total_memo(T))

11 6


# 5. DP từ trên xuống

`top[i][j]` = tổng nhỏ nhất từ đỉnh **tới** ô `(i, j)`. Ô `(i, j)` nhận từ `(i−1, j−1)` hoặc `(i−1, j)`:

```text
top[i][j] = triangle[i][j] + min(top[i-1][j-1], top[i-1][j])
```

**Biên:** `j = 0` chỉ có `top[i-1][0]`; `j = i` chỉ có `top[i-1][i-1]`. Đáp án là `min` của **cả hàng cuối**.

In [5]:
def min_total_top_down(triangle: List[List[int]]) -> int:
    top = [triangle[0][:]]
    for i in range(1, len(triangle)):
        row = []
        for j in range(i + 1):
            cands = []
            if j - 1 >= 0:
                cands.append(top[i - 1][j - 1])
            if j <= i - 1:
                cands.append(top[i - 1][j])
            row.append(triangle[i][j] + min(cands))
        top.append(row)
    return min(top[-1])

print(min_total_top_down(EX), min_total_top_down(T))

11 6


# 6. Ý tưởng chính: DP từ đáy lên

Đi **ngược**: `dp[j]` = tổng nhỏ nhất từ ô `(i, j)` **xuống đáy**. Bắt đầu với `dp = hàng cuối`, rồi lên từng hàng:

```text
dp[j] = triangle[i][j] + min(dp[j], dp[j + 1])      với j = 0 .. i
```

- Mọi ô ở hàng `i` đều có **đủ hai** ô bên dưới → **không có biên**.
- Sau khi xử lý hàng 0, đáp án là `dp[0]`.
- Chỉ cần **một mảng** độ dài `n` → `O(n)` bộ nhớ (follow-up).

Cập nhật `dp[j]` theo thứ tự `j` **tăng dần** là an toàn: `dp[j]` mới chỉ dùng `dp[j]` và `dp[j+1]` cũ, và `dp[j+1]` chưa bị ghi đè.

In [6]:
def min_total_bottom_up(triangle: List[List[int]]) -> int:
    dp = triangle[-1][:]
    for i in range(len(triangle) - 2, -1, -1):
        for j in range(i + 1):
            dp[j] = triangle[i][j] + min(dp[j], dp[j + 1])
    return dp[0]

print(min_total_bottom_up(EX), min_total_bottom_up(T))

11 6


# 7. Mô phỏng

In [7]:
def trace_bottom_up(triangle):
    dp = triangle[-1][:]
    print(f"Bắt đầu (hàng {len(triangle) - 1}): dp = {dp}")
    for i in range(len(triangle) - 2, -1, -1):
        parts = []
        for j in range(i + 1):
            new = triangle[i][j] + min(dp[j], dp[j + 1])
            parts.append(f"{triangle[i][j]}+min({dp[j]},{dp[j + 1]})={new}")
            dp[j] = new
        print(f"Hàng {i}: " + ", ".join(parts) + f"  -> dp[:{i + 1}] = {dp[:i + 1]}")
    print("Đáp án dp[0] =", dp[0])
    return dp[0]

_ = trace_bottom_up(EX)

Bắt đầu (hàng 3): dp = [4, 1, 8, 3]
Hàng 2: 6+min(4,1)=7, 5+min(1,8)=6, 7+min(8,3)=10  -> dp[:3] = [7, 6, 10]
Hàng 1: 3+min(7,6)=9, 4+min(6,10)=10  -> dp[:2] = [9, 10]
Hàng 0: 2+min(9,10)=11  -> dp[:1] = [11]
Đáp án dp[0] = 11


# 8. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def minimumTotal(self, triangle: List[List[int]]) -> int:
        dp = triangle[-1][:]
        for i in range(len(triangle) - 2, -1, -1):
            row = triangle[i]
            for j in range(i + 1):
                dp[j] = row[j] + min(dp[j], dp[j + 1])
        return dp[0]

In [9]:
import itertools

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def brute(triangle):
    n = len(triangle)
    best = float("inf")
    for moves in itertools.product((0, 1), repeat=n - 1):
        j, s = 0, triangle[0][0]
        for i, m in enumerate(moves, start=1):
            j += m
            s += triangle[i][j]
        best = min(best, s)
    return best


sol = Solution()
check("Ví dụ 1", sol.minimumTotal(EX), 11)
check("Ví dụ 2", sol.minimumTotal([[-10]]), -10)
check("Tham lam bị lừa", sol.minimumTotal(T), 6)
check("Số âm", sol.minimumTotal([[-1], [2, 3], [1, -1, -3]]), -1)

random.seed(120)
for _ in range(2000):
    n = random.randint(1, 9)
    tri = [[random.randint(-20, 20) for _ in range(i + 1)] for i in range(n)]
    expected = brute(tri)
    assert sol.minimumTotal(tri) == expected == min_total_top_down(tri) == min_total_memo(tri)
big = [[random.randint(-10**4, 10**4) for _ in range(i + 1)] for i in range(200)]
check("200 hàng", sol.minimumTotal(big) == min_total_top_down(big), True)
print("2000 tam giác ngẫu nhiên: khớp vét cạn mọi 2^(n-1) đường đi")

Ví dụ 1            PASS  11
Ví dụ 2            PASS  -10
Tham lam bị lừa    PASS  6
Số âm              PASS  -1


200 hàng           PASS  True
2000 tam giác ngẫu nhiên: khớp vét cạn mọi 2^(n-1) đường đi


# 9. Vì sao đúng? Độ phức tạp?

**Cấu trúc con tối ưu:** đường tốt nhất từ `(i, j)` xuống đáy đi qua `(i+1, j)` hoặc `(i+1, j+1)`, và phần còn lại phải là đường tốt nhất từ ô đó. Vậy `best(i, j) = triangle[i][j] + min(best(i+1, j), best(i+1, j+1))`.

Đáy lên: tính `best` cho hàng `n−1` (chính là giá trị ô), rồi lần lượt từng hàng phía trên. Ghi đè tại chỗ an toàn vì thứ tự `j` tăng dần.

```text
Time:  O(n²) — tổng số ô n(n+1)/2
Space: O(n)  — một hàng dp
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Tham lam

Như mục 3.

## Lỗi 2: Top-down nhưng lấy `top[-1][-1]` hoặc `top[-1][0]` làm đáp án

Đáp án top-down là `min` của **cả hàng cuối** — đường tốt nhất có thể kết thúc ở bất kỳ ô đáy nào.

In [10]:
def wrong_last_cell(triangle):
    top = [triangle[0][:]]
    for i in range(1, len(triangle)):
        row = []
        for j in range(i + 1):
            cands = [top[i - 1][k] for k in (j - 1, j) if 0 <= k <= i - 1]
            row.append(triangle[i][j] + min(cands))
        top.append(row)
    return top[-1][0]                              # SAI: phải là min(top[-1])

print("Sai :", wrong_last_cell(EX), "| Đúng:", sol.minimumTotal(EX))

Sai : 15 | Đúng: 11


## Lỗi 3: Đáy lên nhưng duyệt `j` giảm dần với một mảng

In [11]:
def wrong_j_desc(triangle):
    dp = triangle[-1][:]
    for i in range(len(triangle) - 2, -1, -1):
        for j in range(i, -1, -1):                 # SAI: ghi đè dp[j + 1] trước khi dùng
            dp[j] = triangle[i][j] + min(dp[j], dp[j + 1])
    return dp[0]

print("j giảm dần:", wrong_j_desc(EX), "| j tăng dần:", sol.minimumTotal(EX))

j giảm dần: 12 | j tăng dần: 11


Kết quả **bị sai** vì khi `j` giảm, `dp[j+1]` đã bị ghi đè thành giá trị của **hàng `i`** trước khi ta dùng nó cho `dp[j]`. Hướng duyệt khi dùng một mảng luôn phải chọn sao cho ô cần đọc **chưa** bị ghi đè.

## Lỗi 4: Sửa trực tiếp `triangle` khi không được phép

Đáy lên có thể làm tại chỗ trên `triangle` (bộ nhớ `O(1)`), nhưng sẽ **phá hủy** input. Chỉ làm khi đề cho phép — ở đây ta sao chép hàng cuối.

# 11. Mẹo nhận diện pattern

- **"Đường đi trên lưới / tam giác, mỗi bước chọn trong vài ô kề"** → DP; thường đi **từ đích ngược về** để bỏ trường hợp biên.
- **Một mảng dp** khi hàng `i` chỉ phụ thuộc hàng `i+1` — chú ý **chiều duyệt** để không ghi đè ô còn cần.

Bài liên quan: **#64 Minimum Path Sum**, **#931 Minimum Falling Path Sum** (mỗi bước 3 ô), **#1289 Minimum Falling Path Sum II** (Hard), **#118 Pascal's Triangle** (cùng hình dạng).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy DP đáy lên cho `[[1], [2, 3], [4, 5, 6], [7, 8, 9, 10]]`.

## Bài 2 — Điền chỗ trống

In [12]:
def exercise_2_minimum_total(triangle: List[List[int]]) -> int:
    dp = triangle[-1][:]
    for i in range(len(triangle) - 2, -1, -1):
        # TODO: cập nhật dp[0..i]
        pass
    return dp[0]


# assert exercise_2_minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == 11
# assert exercise_2_minimum_total([[-10]]) == -10

## Bài 3 — Truy vết đường đi tối ưu

Trả về `(tổng, [chỉ số cột ở mỗi hàng])`. Nếu hòa, ưu tiên cột **bên trái** (`j`).

```python
min_path([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) -> (11, [0, 0, 1, 1])
```

In [13]:
def exercise_3_min_path(triangle: List[List[int]]):
    # TODO
    pass


# assert exercise_3_min_path([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == (11, [0, 0, 1, 1])
# assert exercise_3_min_path([[-10]]) == (-10, [0])

## Bài 4 — Biến thể: Minimum Falling Path Sum (LeetCode #931)

Ma trận vuông `n × n`. Bắt đầu ở bất kỳ ô hàng đầu; mỗi bước xuống hàng dưới ở cột `j − 1`, `j`, hoặc `j + 1`. Tổng nhỏ nhất?

```python
min_falling([[2, 1, 3], [6, 5, 4], [7, 8, 9]]) -> 13      # 1 + 5 + 7  hoặc 1 + 4 + 8
```

In [14]:
def exercise_4_min_falling(matrix: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_min_falling([[2, 1, 3], [6, 5, 4], [7, 8, 9]]) == 13
# assert exercise_4_min_falling([[-19, 57], [-40, -5]]) == -59

## Bài 5 — Mini challenge: tổng lớn nhất, nhưng đổi hướng tối đa `k` lần

Trên tam giác, "đổi hướng" là khi bước hiện tại khác loại với bước trước (xuống-thẳng `j` ↔ xuống-chéo `j + 1`). Tìm tổng **lớn nhất** với **không quá `k`** lần đổi hướng.

Gợi ý nhanh: thêm vào trạng thái **loại bước cuối** và **số lần đã đổi**: `dp[i][j][last][c]`.

In [15]:
def exercise_5_max_with_turns(triangle: List[List[int]], k: int) -> int:
    # TODO
    pass


# assert exercise_5_max_with_turns([[1], [2, 3], [4, 5, 6]], 0) == 10
# assert exercise_5_max_with_turns([[1], [9, 1], [1, 9, 1], [1, 1, 9, 1]], 0) == 12
# assert exercise_5_max_with_turns([[1], [9, 1], [1, 9, 1], [1, 1, 9, 1]], 1) == 28

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_bottom_up` sau khi tự làm.

**Bài 3**
- Mức 1: lưu bảng `best[i][j]` (từ ô xuống đáy) đầy đủ.
- Mức 2: từ `(0, 0)` đi xuống: chọn `j` nếu `best[i+1][j] <= best[i+1][j+1]`, ngược lại `j + 1`.

**Bài 4**
- Mức 1: `dp = matrix[0][:]`; hàng sau: `new[j] = matrix[i][j] + min(dp[j-1], dp[j], dp[j+1])` (bỏ chỉ số ngoài biên).

**Bài 5**
- Mức 1: đệ quy có memo `g(i, j, last, c)` = tổng lớn nhất từ ô `(i, j)` xuống đáy khi bước vừa đi có loại `last` và đã đổi `c` lần.
- Mức 2: ở hàng 0 chưa có bước nào: thử cả hai loại bước đầu mà không tính là đổi hướng.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [16]:
_ = trace_bottom_up([[1], [2, 3], [4, 5, 6], [7, 8, 9, 10]])

Bắt đầu (hàng 3): dp = [7, 8, 9, 10]
Hàng 2: 4+min(7,8)=11, 5+min(8,9)=13, 6+min(9,10)=15  -> dp[:3] = [11, 13, 15]
Hàng 1: 2+min(11,13)=13, 3+min(13,15)=16  -> dp[:2] = [13, 16]
Hàng 0: 1+min(13,16)=14  -> dp[:1] = [14]
Đáp án dp[0] = 14


## Đáp án Bài 2

In [17]:
def answer_2_minimum_total(triangle: List[List[int]]) -> int:
    return min_total_bottom_up(triangle)

assert answer_2_minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == 11
assert answer_2_minimum_total([[-10]]) == -10
print("Bài 2 PASS (giống min_total_bottom_up ở mục 6)")

Bài 2 PASS (giống min_total_bottom_up ở mục 6)


## Đáp án Bài 3

In [18]:
def answer_3_min_path(triangle: List[List[int]]):
    n = len(triangle)
    best = [row[:] for row in triangle]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1):
            best[i][j] = triangle[i][j] + min(best[i + 1][j], best[i + 1][j + 1])
    path, j = [0], 0
    for i in range(1, n):
        if best[i][j + 1] < best[i][j]:
            j += 1
        path.append(j)
    return best[0][0], path

assert answer_3_min_path([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == (11, [0, 0, 1, 1])
assert answer_3_min_path([[-10]]) == (-10, [0])
for _ in range(1000):
    n = random.randint(1, 8)
    tri = [[random.randint(-9, 9) for _ in range(i + 1)] for i in range(n)]
    total, path = answer_3_min_path(tri)
    assert total == sum(tri[i][path[i]] for i in range(n)) == brute(tri)
    assert all(path[i + 1] - path[i] in (0, 1) for i in range(n - 1))
print("Bài 3 PASS")

Bài 3 PASS


Chú ý: vòng truy vết dùng bảng `best` **từ ô xuống đáy** — nên từ đỉnh đi xuống, mỗi bước chọn ô con có `best` nhỏ hơn là chắc chắn nằm trên một đường tối ưu.

## Đáp án Bài 4

In [19]:
def answer_4_min_falling(matrix: List[List[int]]) -> int:
    n = len(matrix)
    dp = matrix[0][:]
    for i in range(1, n):
        new = []
        for j in range(n):
            new.append(matrix[i][j] + min(dp[k] for k in (j - 1, j, j + 1) if 0 <= k < n))
        dp = new
    return min(dp)

assert answer_4_min_falling([[2, 1, 3], [6, 5, 4], [7, 8, 9]]) == 13
assert answer_4_min_falling([[-19, 57], [-40, -5]]) == -59
print("Bài 4 PASS")

Bài 4 PASS


Ở đây cần mảng **mới** (`new`) vì `new[j]` dùng `dp[j−1]` — ô **bên trái** — nên duyệt chiều nào trên một mảng cũng sẽ ghi đè một ô còn cần.

## Đáp án Bài 5

In [20]:
def answer_5_max_with_turns(triangle: List[List[int]], k: int) -> int:
    n = len(triangle)
    @lru_cache(maxsize=None)
    def g(i, j, last, c):
        if i == n - 1:
            return triangle[i][j]
        best = float("-inf")
        for step in (0, 1):
            nc = c + (last is not None and step != last)
            if nc <= k:
                best = max(best, g(i + 1, j + step, step, nc))
        return triangle[i][j] + best
    return g(0, 0, None, 0)

def brute_turns(triangle, k):
    n, best = len(triangle), float("-inf")
    for moves in itertools.product((0, 1), repeat=n - 1):
        turns = sum(a != b for a, b in zip(moves, moves[1:]))
        if turns <= k:
            j, s = 0, triangle[0][0]
            for i, m in enumerate(moves, start=1):
                j += m; s += triangle[i][j]
            best = max(best, s)
    return best

assert answer_5_max_with_turns([[1], [2, 3], [4, 5, 6]], 0) == 10
assert answer_5_max_with_turns([[1], [9, 1], [1, 9, 1], [1, 1, 9, 1]], 0) == 12
assert answer_5_max_with_turns([[1], [9, 1], [1, 9, 1], [1, 1, 9, 1]], 1) == 28
for _ in range(1000):
    n = random.randint(1, 8)
    tri = [[random.randint(-9, 9) for _ in range(i + 1)] for i in range(n)]
    kk = random.randint(0, 3)
    assert answer_5_max_with_turns(tri, kk) == brute_turns(tri, kk)
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> DP từ **đáy lên**: `dp = hàng cuối`; với mỗi hàng `i` từ dưới lên, `dp[j] = tri[i][j] + min(dp[j], dp[j+1])`, `j` tăng dần. Đáp án `dp[0]`. `O(n²)` thời gian, `O(n)` bộ nhớ, không có biên.

---

## Checklist trước khi nộp

- [ ] Không dùng tham lam?
- [ ] Đáy lên: `j` **tăng dần** khi dùng một mảng?
- [ ] Top-down: đáp án là `min` của hàng cuối?
- [ ] Không sửa `triangle` nếu không được phép?